# Clasificación morfológica de Madrid – H3 res.9
**TFM – Álvaro Fernández-Uribarri Poveda · QuEA 2025–26**

Flujo:
1. Descargar red viaria y crear hexágonos H3 res=9
2. Calcular métricas de red por hexágono (Bloque A)
3. Calcular features de catastro — épocas urbanísticas (Bloque B) + morfología construida (Bloque C)
4. Etiquetar automáticamente usando tabla de barrios
5. Entrenar RF + label-cleaning + GBM + selección automática + validación
6. Clasificar todos los hexágonos → exportar GeoPackage v10

In [ ]:
# ── CELDA 1: Imports ───────────────────────────────────────────────────────
import re
import osmnx as ox
import h3
import geopandas as gpd
import pandas as pd
import numpy as np
import json, warnings
from pathlib import Path
from shapely.geometry import Polygon
from scipy.stats import entropy as scipy_entropy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
warnings.filterwarnings('ignore')

ox.settings.log_console = False
ox.settings.use_cache   = True

H3_RES    = 9
CRS       = 'EPSG:25830'
MIN_NODES = 5
UMBRAL    = 0.70

print('OK')

In [ ]:
# ── CELDA 2: Red viaria ────────────────────────────────────────────────────
print('Descargando red viaria...')
G = ox.graph_from_place('Madrid, Spain', network_type='walk')
G = ox.add_edge_bearings(G)          # bearings en WGS84 antes de proyectar
G_proj = ox.project_graph(G, to_crs=CRS)
nodes, edges = ox.graph_to_gdfs(G_proj)
nodes = nodes.reset_index()
edges = edges.reset_index()
edges['length_m'] = edges.geometry.length
print(f'{len(nodes):,} nodos  |  {len(edges):,} aristas')

In [ ]:
# ── CELDA 3: Teselación H3 ────────────────────────────────────────────────
madrid_poly = ox.geocode_to_gdf('Madrid, Spain').to_crs('EPSG:4326')
geom = madrid_poly.geometry.iloc[0]
geojson_dict = json.loads(gpd.GeoSeries([geom]).to_json())
hex_ids = list(h3.geo_to_cells(geojson_dict['features'][0]['geometry'], res=H3_RES))

def h3_to_polygon(h):
    return Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(h)])

hex_gdf = gpd.GeoDataFrame(
    {'hex_id': hex_ids},
    geometry=[h3_to_polygon(h) for h in hex_ids],
    crs='EPSG:4326'
).to_crs(CRS)
hex_gdf['area_km2'] = hex_gdf.geometry.area / 1e6
print(f'Hexágonos H3 res={H3_RES}: {len(hex_gdf):,}')

In [ ]:
# ── CELDA 4: Bloque A — métricas de red viaria (OSMnx) ────────────────────

# -- 4a. Hex_id a cada nodo --------------------------------------------------
nodes_wgs = nodes.to_crs('EPSG:4326').copy()
nodes_wgs['lon'] = nodes_wgs.geometry.x
nodes_wgs['lat'] = nodes_wgs.geometry.y
nodes_wgs['hex_id'] = nodes_wgs.apply(
    lambda r: h3.latlng_to_cell(r['lat'], r['lon'], H3_RES), axis=1
)
G_undir = G_proj.to_undirected()
degree_dict = dict(G_undir.degree())
nodes_wgs['degree'] = nodes_wgs['osmid'].map(degree_dict)

hex_base = hex_gdf[['hex_id', 'geometry']].reset_index(drop=True)

# -- 4b. Métricas por hexágono basadas en nodos ------------------------------
def compute_metrics(group):
    n     = len(group)
    deg   = group['degree']
    dead  = (deg == 1).sum()
    inter = (deg >= 3).sum()
    return pd.Series({
        'n_nodes'         : n,
        'mean_degree'     : deg.mean(),
        'dead_end_ratio'  : dead / n if n > 0 else np.nan,
        'n_intersections' : inter,
        'internal_connect': inter / (inter + dead) if (inter + dead) > 0 else np.nan,
        'degree_std'      : deg.std(),
    })

metrics = nodes_wgs.groupby('hex_id').apply(
    compute_metrics, include_groups=False
).reset_index()

hex_gdf = hex_gdf.merge(metrics, on='hex_id', how='left')
hex_gdf['intersection_dens'] = hex_gdf['n_intersections'] / hex_gdf['area_km2']

# -- 4c. Road density y edge length ------------------------------------------
edges_mid = edges[['geometry', 'length_m']].copy().reset_index(drop=True)
edges_mid['geometry'] = edges_mid.geometry.interpolate(0.5, normalized=True)

joined_e = gpd.sjoin(edges_mid, hex_base, how='left', predicate='within')

road_len = joined_e.groupby('hex_id')['length_m'].sum().reset_index()
road_len.columns = ['hex_id', 'total_road_m']

edge_stats = joined_e.groupby('hex_id')['length_m'].agg(
    edge_len_mean='mean',
    edge_len_std='std'
).reset_index()

hex_gdf = hex_gdf.merge(road_len, on='hex_id', how='left')
hex_gdf = hex_gdf.merge(edge_stats, on='hex_id', how='left')
hex_gdf['road_density_km'] = hex_gdf['total_road_m'] / 1000 / hex_gdf['area_km2']

# -- 4d. Bearing entropy -------------------------------------------------------
def bearing_entropy_hex(bearing_series):
    b = bearing_series.dropna() % 180
    if len(b) < 5:
        return np.nan
    counts, _ = np.histogram(b, bins=18, range=(0, 180))
    total = counts.sum()
    if total == 0:
        return np.nan
    probs = counts / total
    probs = probs[probs > 0]
    return float(-np.sum(probs * np.log(probs)))

if 'bearing' in edges.columns:
    edges_bear = edges[['geometry', 'bearing']].copy().reset_index(drop=True)
    edges_bear['geometry'] = edges_bear.geometry.interpolate(0.5, normalized=True)
    joined_b = gpd.sjoin(edges_bear, hex_base, how='left', predicate='within')
    bear_stats = joined_b.groupby('hex_id')['bearing'].agg(
        bearing_entropy=bearing_entropy_hex,
        bearing_mean='mean'
    ).reset_index()
    hex_gdf = hex_gdf.merge(bear_stats, on='hex_id', how='left')
    print('Bearing entropy: OK')
else:
    hex_gdf['bearing_entropy'] = np.nan
    hex_gdf['bearing_mean']    = np.nan
    print('AVISO: columna "bearing" no encontrada.')

# -- 4e. Circuity ---------------------------------------------------------------
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

nodes_coords = nodes_wgs[['osmid', 'lat', 'lon']].copy()
edges_circ = edges[['u', 'v', 'length_m', 'geometry']].copy().reset_index(drop=True)
edges_circ = edges_circ.merge(
    nodes_coords.rename(columns={'osmid': 'u', 'lat': 'lat_u', 'lon': 'lon_u'}),
    on='u', how='left'
).merge(
    nodes_coords.rename(columns={'osmid': 'v', 'lat': 'lat_v', 'lon': 'lon_v'}),
    on='v', how='left'
)

mask_valid = (
    edges_circ['lat_u'].notna() & edges_circ['lat_v'].notna() &
    (edges_circ['length_m'] > 0)
)
edges_circ.loc[mask_valid, 'straight_m'] = haversine_m(
    edges_circ.loc[mask_valid, 'lat_u'].values,
    edges_circ.loc[mask_valid, 'lon_u'].values,
    edges_circ.loc[mask_valid, 'lat_v'].values,
    edges_circ.loc[mask_valid, 'lon_v'].values,
)
edges_circ['circuity'] = np.where(
    edges_circ['straight_m'] > 1,
    edges_circ['length_m'] / edges_circ['straight_m'],
    np.nan
)
edges_circ['circuity'] = edges_circ['circuity'].clip(upper=5)

edges_circ_geo = gpd.GeoDataFrame(
    edges_circ[['circuity', 'geometry']].copy(), crs=CRS
)
edges_circ_geo['geometry'] = edges_circ_geo.geometry.interpolate(0.5, normalized=True)
joined_c = gpd.sjoin(edges_circ_geo, hex_base, how='left', predicate='within')
circ_stats = joined_c.groupby('hex_id')['circuity'].agg(
    mean_circuity='mean',
    p90_circuity=lambda x: np.nanpercentile(x, 90)
).reset_index()

hex_gdf = hex_gdf.merge(circ_stats, on='hex_id', how='left')
print('Circuity: OK')

# -- 4f. Filtro mínimo y lista de features Bloque A --------------------------
hex_gdf = hex_gdf[hex_gdf['n_nodes'] >= MIN_NODES].copy().reset_index(drop=True)
hex_gdf['tipologia_train'] = np.nan

FEATURES_A = [
    'mean_degree',
    'dead_end_ratio',
    'intersection_dens',
    'road_density_km',
    'internal_connect',
    'degree_std',
    'edge_len_mean',
    'edge_len_std',
    'bearing_entropy',
    'mean_circuity',
    'p90_circuity',
]

print(f'\nHexágonos válidos: {len(hex_gdf):,}')
print(f'Bloque A ({len(FEATURES_A)} features): {FEATURES_A}')

## Celda 4b: Bloques B y C — datos del Catastro

**Bloque B — épocas urbanísticas** (cortes teóricos del TFM):
- `cat_pct_pre1860` — núcleo medieval / crecimiento orgánico
- `cat_pct_1860_1940` — Plan Castro / ensanche clásico
- `cat_pct_1940_1960` — posguerra, crecimiento espontáneo
- `cat_pct_1960_1985` — expansión urbana planificada
- `cat_pct_post1985` — PAUs (PGOUM 1985, 1997)
- `cat_year_median` — época constructiva dominante
- `cat_year_iqr` — mezcla temporal (proxy de espontaneidad)

**Bloque C — morfología construida**:
- `cat_footprint_ratio` — compacidad: huella edificada / área hexágono
- `cat_far` — Floor Area Ratio: m² construidos / m² suelo
- `cat_floors_mean` — altura media en plantas
- `cat_floors_cv` — heterogeneidad vertical (CV alto → espontáneo, CV bajo → PAU)
- `cat_pct_residential` — porcentaje de uso residencial

In [ ]:
# ── CELDA 4b: Bloques B + C — catastro ────────────────────────────────────
# Columnas del shapefile INSPIRE de Catastro que se usan:
#   beginning   → año construcción  (formato '1965-01-01' o '1965')
#   currentUse  → uso del edificio
#   numberOfFl  → número de plantas (puede llegar como 'B+3', '04', etc.)
#   officialAr  → ETIQUETA de tipo ('grossFloorArea'), NO el valor numérico
#   value       → área construida real en m² (grossFloorArea) — 100% cobertura
#   geometry    → polígono del edificio (para footprint)

CATASTRO_PATH = Path(r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\Catastro\catastro_comunidad_de_madrid.shp')

print(f'Cargando catastro desde: {CATASTRO_PATH.resolve()}')
cat = gpd.read_file(CATASTRO_PATH)
print(f'  Edificios CM total: {len(cat):,}  |  CRS: {cat.crs}')
print(f'  Columnas: {list(cat.columns)}')

# ── Recortar a Madrid capital ─────────────────────────────────────────────
madrid_bbox = hex_gdf.to_crs('EPSG:4326').total_bounds
cat_wgs     = cat.to_crs('EPSG:4326')
cat_madrid  = cat_wgs.cx[madrid_bbox[0]:madrid_bbox[2],
                          madrid_bbox[1]:madrid_bbox[3]].copy()
cat_madrid  = cat_madrid.to_crs(CRS)
print(f'  Edificios en Madrid capital: {len(cat_madrid):,}')

# ── Parsear año de construcción ───────────────────────────────────────────
def parse_year(val):
    if pd.isna(val):
        return np.nan
    try:
        return int(str(val).strip()[:4])
    except:
        return np.nan

cat_madrid['year_built'] = cat_madrid['beginning'].apply(parse_year)
mask_ok = cat_madrid['year_built'].between(1700, 2025)
cat_madrid.loc[~mask_ok, 'year_built'] = np.nan
print(f'  Años válidos (1700-2025): {mask_ok.sum():,} ({mask_ok.mean()*100:.1f}%)')

# ── Área construida ───────────────────────────────────────────────────────
# CORRECCIÓN: officialAr es siempre la etiqueta 'grossFloorArea' (texto).
# El valor numérico real en m² está en la columna 'value' (100% cobertura).
cat_madrid['footprint_m2']      = cat_madrid.geometry.area
cat_madrid['area_construida_m2'] = pd.to_numeric(cat_madrid['value'], errors='coerce')
pct_ar = cat_madrid['area_construida_m2'].notna().mean() * 100
print(f'  Área construida (value): {pct_ar:.1f}% cobertura')

# Plantas estimadas: grossFloorArea / footprint (proxy de intensidad vertical)
# No son plantas oficiales — es un ratio de edificabilidad ≈ número de plantas
cat_madrid['floors_est'] = cat_madrid['area_construida_m2'] / cat_madrid['footprint_m2']
# Recortar outliers: valores fuera de [0.5, 30] se descartan
cat_madrid.loc[~cat_madrid['floors_est'].between(0.5, 30), 'floors_est'] = np.nan
pct_fl = cat_madrid['floors_est'].notna().mean() * 100
print(f'  Plantas estimadas (value/footprint): {pct_fl:.1f}% cobertura')

# ── Parsear uso ────────────────────────────────────────────────────────────
def parse_use(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if s.startswith('1') or 'resid' in s:                    return 1
    if s.startswith('5') or 'retail' in s or 'commerc' in s: return 5
    if s.startswith('3') or 'indust' in s:                   return 3
    if s.startswith('4') or 'offic' in s:                    return 4
    if s.startswith('6') or 'public' in s:                   return 6
    if s.startswith('2') or 'agr' in s:                      return 2
    return np.nan

cat_madrid['use_code'] = cat_madrid['currentUse'].apply(parse_use)

# ── Spatial join: centroide → hexágono ────────────────────────────────────
cat_pts = cat_madrid[[
    'year_built', 'use_code', 'floors_est',
    'footprint_m2', 'area_construida_m2', 'geometry'
]].copy()
cat_pts['geometry'] = cat_pts.geometry.centroid

joined_cat = gpd.sjoin(
    cat_pts,
    hex_gdf[['hex_id', 'geometry', 'area_km2']],
    how='left', predicate='within'
)
print(f'  Edificios con hex asignado: {joined_cat["hex_id"].notna().sum():,}')

# ── Agregación por hexágono ───────────────────────────────────────────────
def cat_features_v2(g):
    yr  = g['year_built'].dropna()
    use = g['use_code'].dropna()
    fl  = g['floors_est'].dropna()
    fp  = g['footprint_m2'].dropna()
    ar  = g['area_construida_m2'].dropna()
    n   = len(g)
    hex_area_m2 = float(g['area_km2'].iloc[0]) * 1e6

    # ── BLOQUE B: épocas urbanísticas con cortes teóricos ──────────────────
    pct_pre1860   = (yr < 1860).mean()                          if len(yr) > 0 else np.nan
    pct_1860_1940 = ((yr >= 1860) & (yr < 1940)).mean()         if len(yr) > 0 else np.nan
    pct_1940_1960 = ((yr >= 1940) & (yr < 1960)).mean()         if len(yr) > 0 else np.nan
    pct_1960_1985 = ((yr >= 1960) & (yr < 1985)).mean()         if len(yr) > 0 else np.nan
    pct_post1985  = (yr >= 1985).mean()                         if len(yr) > 0 else np.nan
    yr_median     = yr.median()                                 if len(yr) > 0 else np.nan
    yr_iqr        = (yr.quantile(0.75) - yr.quantile(0.25))     if len(yr) > 0 else np.nan

    # ── BLOQUE C: morfología construida ────────────────────────────────────
    footprint_ratio = fp.sum() / hex_area_m2 if (len(fp) > 0 and hex_area_m2 > 0) else np.nan

    # FAR correcto: usa value (grossFloorArea) — no footprint*1 como antes
    far = ar.sum() / hex_area_m2 if (len(ar) > 0 and hex_area_m2 > 0) else np.nan

    # Plantas estimadas por catastro (value/footprint por edificio, luego agregado)
    floors_mean = fl.mean()                                     if len(fl) >= 3 else np.nan
    floors_cv   = (fl.std() / fl.mean())                       if (len(fl) >= 3 and fl.mean() > 0) else np.nan

    # % uso residencial
    pct_residential = (use == 1).mean()                        if len(use) > 0 else np.nan

    return pd.Series({
        # Bloque B
        'cat_pct_pre1860'    : pct_pre1860,
        'cat_pct_1860_1940'  : pct_1860_1940,
        'cat_pct_1940_1960'  : pct_1940_1960,
        'cat_pct_1960_1985'  : pct_1960_1985,
        'cat_pct_post1985'   : pct_post1985,
        'cat_year_median'    : yr_median,
        'cat_year_iqr'       : yr_iqr,
        # Bloque C
        'cat_footprint_ratio': footprint_ratio,
        'cat_far'            : far,
        'cat_floors_mean'    : floors_mean,
        'cat_floors_cv'      : floors_cv,
        'cat_pct_residential': pct_residential,
        # Auxiliar
        'cat_n_buildings'    : n,
    })

cat_agg = joined_cat.groupby('hex_id').apply(
    cat_features_v2, include_groups=False
).reset_index()

# ── Merge con hex_gdf ─────────────────────────────────────────────────────
hex_gdf = hex_gdf.merge(cat_agg, on='hex_id', how='left')

# ── Construir lista FEATURES completa ────────────────────────────────────
FEATURES_B = [
    'cat_pct_pre1860',
    'cat_pct_1860_1940',
    'cat_pct_1940_1960',
    'cat_pct_1960_1985',
    'cat_pct_post1985',
    'cat_year_median',
    'cat_year_iqr',
]

FEATURES_C = [
    'cat_footprint_ratio',
    'cat_far',
    'cat_floors_mean',
    'cat_floors_cv',
    'cat_pct_residential',
]

FEATURES = FEATURES_A + FEATURES_B + FEATURES_C

# ── Diagnóstico de cobertura ──────────────────────────────────────────────
n_hex = len(hex_gdf)
print(f'\nCobertura — Bloque B (épocas urbanísticas):')
for col in FEATURES_B:
    n_ok = hex_gdf[col].notna().sum()
    print(f'  {col}: {n_ok:,}/{n_hex:,} ({n_ok/n_hex*100:.1f}%)')

print(f'\nCobertura — Bloque C (morfología construida):')
for col in FEATURES_C:
    n_ok = hex_gdf[col].notna().sum()
    print(f'  {col}: {n_ok:,}/{n_hex:,} ({n_ok/n_hex*100:.1f}%)')

print(f'\nFEATURES totales ({len(FEATURES)}):')
print(f'  Bloque A ({len(FEATURES_A)}): {FEATURES_A}')
print(f'  Bloque B ({len(FEATURES_B)}): {FEATURES_B}')
print(f'  Bloque C ({len(FEATURES_C)}): {FEATURES_C}')

In [ ]:
# ── CELDA 5: Etiquetar por barrios ─────────────────────────────────────────
BARRIOS_LABEL = {
    # PLANIFICADO (1)
    'Recoletos': 1, 'Goya': 1, 'Fuente del Berro': 1, 'Guindalera': 1,
    'Lista': 1, 'Castellana': 1,
    'Gaztambide': 1, 'Arapiles': 1, 'Trafalgar': 1, 'Almagro': 1,
    'Ríos Rosas': 1, 'Vallehermoso': 1,
    'El Retiro': 1, 'Ibiza': 1, 'Jerónimos': 1, 'Niño Jesús': 1,
    'Adelfas': 1, 'Estrella': 1,
    'El Viso': 1, 'Prosperidad': 1, 'Ciudad Jardín': 1,
    'Hispanoamérica': 1, 'Nueva España': 1, 'Castilla': 1,
    'Imperial': 1, 'Acacias': 1, 'Chopera': 1, 'Legazpi': 1,
    'Delicias': 1, 'Palos de Moguer': 1,
    'Pavones': 1, 'Horcajo': 1, 'Marroquina': 1, 'Media Legua': 1,
    'Fontarrón': 1, 'Vinateros': 1,
    'Sanchinarro': 1, 'Las Tablas': 1, 'Valdebebas': 1,
    'Campo de las Naciones': 1, 'Ensanche de Vallecas': 1,
    'Valdebernardo': 1, 'Arcos': 1, 'Rosas': 1, 'Rejas': 1,
    'Simancas': 1, 'Alameda de Osuna': 1, 'Corralejos': 1,
    'Argüelles': 1, 'Ciudad Universitaria': 1, 'San Pascual': 1,
    'Costillares': 1, 'Concepción': 1, 'La Paz': 1,
    # ESPONTÁNEO (0)
    'Palacio': 0, 'Embajadores': 0, 'Cortes': 0, 'Justicia': 0,
    'Universidad': 0, 'Sol': 0,
    'Los Cármenes': 0, 'Puerta del Ángel': 0, 'Lucero': 0,
    'Cuatro Vientos': 0, 'Las Águilas': 0,
    'Comillas': 0, 'Opañel': 0, 'San Isidro': 0, 'Vista Alegre': 0,
    'Pradolongo': 0, 'Moscardó': 0, 'Zofío': 0, 'Buenavista': 0,
    'Bellas Vistas': 0, 'Castillejos': 0, 'Almenara': 0,
    'Valdeacederas': 0, 'Berruguete': 0,
    'San Andrés': 0, 'San Cristóbal': 0, 'Los Ángeles': 0,
    'Fuencarral': 0, 'El Pardo': 0, 'Aravaca': 0, 'Ventas': 0,
    'Pueblo Nuevo': 0, 'Quintana': 0, 'Pinar del Rey': 0,
    'Canillas': 0, 'Canillejas': 0,
    'Casco Histórico de Vallecas': 0,
    'Casco Histórico de Vicálvaro': 0,
    'Casco Histórico de Barajas': 0,
    # EXCLUIR (-1) — morfología ambigua
    'Aluche': -1, 'Campamento': -1, 'PAU de Carabanchel': -1,
    'Orcasitas': -1, 'Butarque': -1, 'Los Rosales': -1,
    'Cuatro Caminos': -1, 'Azca': -1,
    'Palomeras Altas': -1, 'Palomeras Bajas': -1,
    'Valdemarín': -1, 'El Plantío': -1, 'Cañaveral': -1,
}

print('Descargando barrios...')
barrios = ox.features_from_place('Madrid, Spain', tags={'admin_level': '10'})
barrios = barrios[
    barrios.geometry.type.isin(['Polygon', 'MultiPolygon'])
][['name', 'geometry']].to_crs(CRS).reset_index(drop=True)

barrios['label'] = barrios['name'].map(BARRIOS_LABEL)
barrios_validos = barrios[barrios['label'].notna()].copy()
print(f'Barrios encontrados en OSM: {len(barrios)}')
print(f'Barrios con etiqueta: {len(barrios_validos)}')
print(f'  Planificado (1): {(barrios_validos["label"]==1).sum()}')
print(f'  Espontáneo  (0): {(barrios_validos["label"]==0).sum()}')
print(f'  Excluidos  (-1): {(barrios_validos["label"]==-1).sum()}')

In [ ]:
# ── CELDA 6: Spatial join barrios → hexágonos ─────────────────────────────
hex_centroids = hex_gdf[['hex_id','geometry']].copy()
hex_centroids['geometry'] = hex_gdf.geometry.centroid

joined_barrios = gpd.sjoin(
    hex_centroids,
    barrios_validos[['name', 'label', 'geometry']],
    how='left',
    predicate='within'
)

hex_gdf = hex_gdf.merge(
    joined_barrios[['hex_id', 'label', 'name']].rename(
        columns={'label': 'barrio_label', 'name': 'barrio_nombre'}
    ),
    on='hex_id', how='left'
)

hex_gdf['tipologia_train'] = np.where(
    hex_gdf['barrio_label'].isin([0, 1]),
    hex_gdf['barrio_label'],
    np.nan
)

n_train = hex_gdf['tipologia_train'].notna().sum()
n_excl  = (hex_gdf['barrio_label'] == -1).sum()
n_nan   = hex_gdf['barrio_label'].isna().sum()
print(f'Hexágonos para training: {n_train}')
print(f'  Planificado: {(hex_gdf["tipologia_train"]==1).sum()}')
print(f'  Espontáneo:  {(hex_gdf["tipologia_train"]==0).sum()}')
print(f'Excluidos (mixtos): {n_excl}')
print(f'Sin barrio asignado: {n_nan}')

In [ ]:
# ── CELDA 7: Visualizar training set ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 12))
hex_gdf[hex_gdf['tipologia_train'] == 0].plot(
    ax=ax, color='#E8735A', alpha=0.7, linewidth=0.2, edgecolor='white', label='Espontáneo (train)')
hex_gdf[hex_gdf['tipologia_train'] == 1].plot(
    ax=ax, color='#4A90C4', alpha=0.7, linewidth=0.2, edgecolor='white', label='Planificado (train)')
hex_gdf[hex_gdf['barrio_label'] == -1].plot(
    ax=ax, color='#AAAAAA', alpha=0.4, linewidth=0.2, edgecolor='white', label='Excluido')
hex_gdf[hex_gdf['barrio_label'].isna()].plot(
    ax=ax, color='#EEEEEE', alpha=0.3, linewidth=0.2, edgecolor='white', label='Sin etiquetar')
ax.legend(fontsize=11)
ax.set_title(f'Training set — {n_train} hexágonos etiquetados', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('training_set_madrid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELDA 8: RF + label-cleaning 0.60 + GBM + selección automática ────────

# -- 8a. Training set inicial ------------------------------------------------
feature_cols = [c for c in FEATURES if c in hex_gdf.columns]
missing_features = [c for c in FEATURES if c not in hex_gdf.columns]
if missing_features:
    print(f'AVISO — features no encontradas en hex_gdf: {missing_features}')

train_raw = hex_gdf.dropna(subset=['tipologia_train']).copy()
if len(train_raw) == 0:
    raise ValueError("No hay observaciones con 'tipologia_train' no nulo.")

X_df = train_raw[feature_cols].copy()

# Quitar features completamente vacías
all_nan_cols = X_df.columns[X_df.isna().all()].tolist()
if all_nan_cols:
    print(f'Features eliminadas (completamente vacías): {all_nan_cols}')
    X_df = X_df.drop(columns=all_nan_cols)

feature_cols_final = X_df.columns.tolist()
if len(feature_cols_final) == 0:
    raise ValueError('No queda ninguna feature útil para entrenar.')

X_df = X_df.apply(pd.to_numeric, errors='coerce')
medians = X_df.median()
X_df = X_df.fillna(medians)

still_bad = X_df.columns[X_df.isna().all()].tolist()
if still_bad:
    X_df = X_df.drop(columns=still_bad)
    feature_cols_final = X_df.columns.tolist()

y_raw = train_raw['tipologia_train'].astype(int)
X_raw = X_df

print(f'Training set inicial: {len(train_raw):,} '
      f'({(y_raw==0).sum()} espontáneo, {(y_raw==1).sum()} planificado)')
print(f'Features usadas: {len(feature_cols_final)}')

class_counts = y_raw.value_counts().sort_index()
min_class_count = class_counts.min()
if len(class_counts) < 2:
    raise ValueError(f'Solo hay una clase en y_raw: {class_counts.to_dict()}')
if min_class_count < 2:
    raise ValueError(f'Muy pocas observaciones en una clase: {class_counts.to_dict()}')

n_splits = min(5, min_class_count)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# -- 8b. RF base para detectar hexágonos ruidosos ---------------------------
rf_base = RandomForestClassifier(
    n_estimators=500, min_samples_leaf=3, max_features='sqrt',
    class_weight='balanced', random_state=42, n_jobs=-1
)
proba_cv  = cross_val_predict(rf_base, X_raw, y_raw, cv=skf, method='predict_proba')
confianza = proba_cv.max(axis=1)

# -- 8c. Label-cleaning conservador (0.60) ----------------------------------
UMBRAL_CONFIANZA = 0.60
mask_clean = confianza >= UMBRAL_CONFIANZA
X_clean = X_raw.loc[mask_clean].copy()
y_clean = y_raw.loc[mask_clean].copy()

n_excl_lc = (~mask_clean).sum()
print(f'\nLabel-cleaning (umbral={UMBRAL_CONFIANZA}):')
print(f'  Excluidos: {n_excl_lc:,} ({n_excl_lc/len(y_raw)*100:.1f}%)')
print(f'  Training limpio: {len(X_clean):,} '
      f'({(y_clean==0).sum()} espontáneo, {(y_clean==1).sum()} planificado)')

class_counts_clean = y_clean.value_counts().sort_index()
if len(class_counts_clean) < 2:
    raise ValueError(f'Tras label-cleaning solo queda una clase: {class_counts_clean.to_dict()}')
min_class_count_clean = class_counts_clean.min()
if min_class_count_clean < 2:
    raise ValueError(f'Tras label-cleaning hay muy pocas obs. por clase: {class_counts_clean.to_dict()}')

n_splits_clean = min(5, min_class_count_clean)
skf_clean = StratifiedKFold(n_splits=n_splits_clean, shuffle=True, random_state=42)

# -- 8d. Random Forest final ------------------------------------------------
rf = RandomForestClassifier(
    n_estimators=600, min_samples_leaf=3, max_features='sqrt',
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_clean, y_clean)

# -- 8e. Gradient Boosting --------------------------------------------------
gbm = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    min_samples_leaf=10, subsample=0.8, random_state=42
)
gbm.fit(X_clean, y_clean)

# -- 8f. Comparación CV -----------------------------------------------------
results = {}
for name, model in [('Random Forest', rf), ('Gradient Boosting', gbm)]:
    cv_f1  = cross_val_score(model, X_clean, y_clean, cv=skf_clean, scoring='f1_macro')
    cv_acc = cross_val_score(model, X_clean, y_clean, cv=skf_clean, scoring='accuracy')
    results[name] = cv_acc.mean()
    print(f'\n── {name} ──')
    print(f'  F1 macro : {cv_f1.mean():.3f} +- {cv_f1.std():.3f}')
    print(f'  Accuracy : {cv_acc.mean():.3f} +- {cv_acc.std():.3f}')

# -- 8g. Seleccionar mejor modelo -------------------------------------------
best_name  = max(results, key=results.get)
best_model = rf if best_name == 'Random Forest' else gbm
print(f'\n→ Modelo seleccionado: {best_name} ({results[best_name]:.3f})')
rf = best_model

# -- 8h. Importancia de features --------------------------------------------
feat_imp = pd.Series(
    rf.feature_importances_, index=feature_cols_final
).sort_values(ascending=False)
print('\nImportancia de variables:')
print(feat_imp.round(3).to_string())

# -- 8i. Matriz de confusión ------------------------------------------------
y_pred_cv = cross_val_predict(rf, X_clean, y_clean, cv=skf_clean)
print(f'\nClassification report (CV {n_splits_clean}-fold · {best_name}):')
print(classification_report(y_clean, y_pred_cv,
                             target_names=['Espontáneo', 'Planificado']))

cm   = confusion_matrix(y_clean, y_pred_cv)
disp = ConfusionMatrixDisplay(cm, display_labels=['Espontáneo', 'Planificado'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax)
ax.set_title(f'Matriz de confusión (CV {n_splits_clean}-fold · v10 · {best_name})')
plt.tight_layout()
plt.savefig('confusion_matrix_v10.png', dpi=150, bbox_inches='tight')
plt.show()

# -- 8j. Auditoría label-cleaning -------------------------------------------
train_raw = train_raw.copy()
train_raw['confianza_rf']  = confianza
train_raw['label_excluida'] = ~mask_clean
hex_gdf = hex_gdf.merge(
    train_raw[['hex_id', 'confianza_rf', 'label_excluida']],
    on='hex_id', how='left'
)

In [ ]:
# Diagnóstico rápido — ejecuta esto antes de tocar el notebook
tags = {'building': True}
buildings_osm = ox.features_from_place('Madrid, Spain', tags=tags)
buildings_osm = buildings_osm[
    buildings_osm.geometry.type.isin(['Polygon','MultiPolygon'])
].copy()

total = len(buildings_osm)
con_levels = buildings_osm['building:levels'].notna().sum() if 'building:levels' in buildings_osm.columns else 0

print(f"Edificios OSM total:         {total:,}")
print(f"Con building:levels:         {con_levels:,} ({con_levels/total*100:.1f}%)")
print(f"\nDistribución de levels:")
if con_levels > 0:
    niveles = pd.to_numeric(buildings_osm['building:levels'], errors='coerce').dropna()
    print(niveles.describe())
    print(f"\nValores más frecuentes:")
    print(niveles.value_counts().head(10))

In [ ]:
# ── CELDA 9: Clasificar todos los hexágonos ────────────────────────────────
if 'feature_cols_final' not in globals():
    raise ValueError("Ejecuta antes la celda 8.")
if 'medians' not in globals():
    raise ValueError("Ejecuta antes la celda 8.")

X_all = hex_gdf[feature_cols_final].copy()
X_all = X_all.apply(pd.to_numeric, errors='coerce')
X_all = X_all.fillna(medians[feature_cols_final])

if hasattr(rf, 'feature_names_in_'):
    X_all = X_all[list(rf.feature_names_in_)]

probas = rf.predict_proba(X_all)

all_valid = hex_gdf.copy()
all_valid['proba_espontaneo']  = probas[:, 0]
all_valid['proba_planificado'] = probas[:, 1]
all_valid['proba_max']         = probas.max(axis=1)
all_valid['pred_rf']           = rf.predict(X_all)

UMBRAL_MAPA = 0.55
all_valid['tipologia_label'] = all_valid.apply(
    lambda r: ('Espontáneo' if r['pred_rf'] == 0 else 'Planificado')
              if r['proba_max'] >= UMBRAL_MAPA else 'Transición',
    axis=1
)

# Variable continua para la regresión (0 = espontáneo puro, 1 = planificado puro)
all_valid['morfologia_continua'] = all_valid['proba_planificado']

print(f'Hexágonos clasificados: {len(all_valid):,}')
print(f'UMBRAL_MAPA: {UMBRAL_MAPA}')
print(all_valid['tipologia_label'].value_counts().to_string())
print(f'\nVariable continua morfologia_continua: '
      f'media={all_valid["morfologia_continua"].mean():.3f}, '
      f'std={all_valid["morfologia_continua"].std():.3f}')

In [ ]:
# ── CELDA 10: Mapa RF puro ────────────────────────────────────────────────
COLORES_MAP = {'Espontáneo': '#E8735A', 'Planificado': '#4A90C4', 'Transición': '#D0D0D0'}

fig, ax = plt.subplots(figsize=(13, 13))
for label, color in COLORES_MAP.items():
    all_valid[all_valid['tipologia_label'] == label].plot(
        ax=ax, color=color, alpha=0.80, linewidth=0.2, edgecolor='white'
    )
patches = [mpatches.Patch(color=c, label=l) for l, c in COLORES_MAP.items()]
ax.legend(handles=patches, loc='lower right', fontsize=12)

counts = all_valid['tipologia_label'].value_counts()
ax.set_title(
    f'Clasificación morfológica (RF v10) – H3 res.{H3_RES}\n'
    f'Espontáneo: {counts.get("Espontáneo",0):,}  |  '
    f'Planificado: {counts.get("Planificado",0):,}  |  '
    f'Transición: {counts.get("Transición",0):,}',
    fontsize=12
)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('clasificacion_morfologica_madrid_v10.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELDA 10b: Mapa RF con límites de barrios ─────────────────────────────
fig, ax = plt.subplots(figsize=(13, 13))
for label, color in COLORES_MAP.items():
    all_valid[all_valid['tipologia_label'] == label].plot(
        ax=ax, color=color, alpha=0.80, linewidth=0.2, edgecolor='white'
    )
barrios.boundary.plot(ax=ax, color='black', linewidth=0.5, alpha=0.45)
patches = [mpatches.Patch(color=c, label=l) for l, c in COLORES_MAP.items()]
ax.legend(handles=patches, loc='lower right', fontsize=12)
counts = all_valid['tipologia_label'].value_counts()
ax.set_title(
    f'Clasificación morfológica (RF v10) + límites de barrios – H3 res.{H3_RES}\n'
    f'Espontáneo: {counts.get("Espontáneo",0):,}  |  '
    f'Planificado: {counts.get("Planificado",0):,}  |  '
    f'Transición: {counts.get("Transición",0):,}',
    fontsize=12
)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('clasificacion_morfologica_madrid_con_barrios_v10.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELDA 10c: Mapa interactivo con zoom real (Folium) ────────────────────
%pip install -q folium mapclassify
import folium

all_valid_wgs = all_valid.to_crs(epsg=4326)
centro = [
    all_valid_wgs.geometry.centroid.y.mean(),
    all_valid_wgs.geometry.centroid.x.mean()
]

m = folium.Map(location=centro, zoom_start=11, tiles='CartoDB positron')

color_map = {
    'Espontáneo': '#E8735A',
    'Planificado': '#4A90C4',
    'Transición':  '#D0D0D0'
}

folium.GeoJson(
    all_valid_wgs[['tipologia_label', 'hex_id', 'morfologia_continua',
                   'proba_max', 'geometry']].to_json(),
    style_function=lambda feature: {
        'fillColor'  : color_map.get(feature['properties']['tipologia_label'], '#999999'),
        'color'      : 'white',
        'weight'     : 0.3,
        'fillOpacity': 0.65
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['tipologia_label', 'proba_max', 'morfologia_continua'],
        aliases=['Tipología', 'Confianza RF', 'Morfología continua']
    )
).add_to(m)

m

In [ ]:
# ── CELDA 11: Mapa respetando etiquetas originales del training set ────────
all_valid2 = all_valid.copy()

mask_train = all_valid2['tipologia_train'].notna()
all_valid2.loc[mask_train & (all_valid2['tipologia_train'] == 0), 'tipologia_label'] = 'Espontáneo'
all_valid2.loc[mask_train & (all_valid2['tipologia_train'] == 1), 'tipologia_label'] = 'Planificado'

fig, ax = plt.subplots(figsize=(13, 13))
for label, color in COLORES_MAP.items():
    all_valid2[all_valid2['tipologia_label'] == label].plot(
        ax=ax, color=color, alpha=0.80, linewidth=0.2, edgecolor='white'
    )
patches = [mpatches.Patch(color=c, label=l) for l, c in COLORES_MAP.items()]
ax.legend(handles=patches, loc='lower right', fontsize=12)
counts2 = all_valid2['tipologia_label'].value_counts()
ax.set_title(
    f'Clasificación morfológica (etiquetas originales respetadas) – H3 res.{H3_RES}\n'
    f'Espontáneo: {counts2.get("Espontáneo",0):,}  |  '
    f'Planificado: {counts2.get("Planificado",0):,}  |  '
    f'Transición: {counts2.get("Transición",0):,}',
    fontsize=12
)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('clasificacion_morfologica_madrid_v10b.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELDA 12: Exportar GeoPackage v10 ─────────────────────────────────────
cols = [
    # Identificadores y tipología
    'hex_id', 'barrio_nombre', 'barrio_label', 'tipologia_train',
    'pred_rf', 'tipologia_label', 'proba_max',
    'proba_espontaneo', 'proba_planificado', 'morfologia_continua',
    'confianza_rf', 'label_excluida',
    # Bloque A — red viaria
    'n_nodes', 'mean_degree', 'dead_end_ratio',
    'intersection_dens', 'road_density_km', 'internal_connect',
    'degree_std', 'edge_len_mean', 'edge_len_std',
    'bearing_entropy', 'mean_circuity', 'p90_circuity',
    # Bloque B — épocas urbanísticas
    'cat_pct_pre1860', 'cat_pct_1860_1940', 'cat_pct_1940_1960',
    'cat_pct_1960_1985', 'cat_pct_post1985',
    'cat_year_median', 'cat_year_iqr',
    # Bloque C — morfología construida
    'cat_footprint_ratio', 'cat_far',
    'cat_floors_mean', 'cat_floors_cv',
    'cat_pct_residential',
    # Auxiliar
    'cat_n_buildings', 'area_km2',
    # Geometría
    'geometry'
]

out_cols = [c for c in cols if c in all_valid2.columns]
all_valid2[out_cols].to_file('madrid_morfologia_h3_v10.gpkg', driver='GPKG')
print(f'Exportado: madrid_morfologia_h3_v10.gpkg')
print(f'Columnas exportadas ({len(out_cols)}): {out_cols}')